# IMC2024 ver86

**Minimum verified matches 50**

Matcher density or edge acceptance sweep on the full ver28 path.


In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/xdg-cache')

MODULES = [
    'pycolmap', 'h5py', 'kornia', 'kornia_rs', 'kneed', 'lightglue',
    'omegaconf', 'antlr4', 'yaml', 'sklearn',
]
PIP_PACKAGES = [
    'pycolmap', 'h5py', 'kornia', 'kornia-rs', 'kneed', 'lightglue',
    'omegaconf', 'antlr4-python3-runtime', 'PyYAML', 'typing-extensions',
]

def has_module(name):
    return importlib.util.find_spec(name) is not None

def existing_deps_root():
    input_root = Path('/kaggle/input')
    preferred = [
        Path('/kaggle/input/imc2024-final-project-deps'),
        Path('/kaggle/input/datasets/milktea7654/imc2024-final-project-deps'),
        Path('/kaggle/input/imc2024-final-project-deps-ver9'),
        Path('/kaggle/input/datasets/milktea7654/imc2024-final-project-deps-ver9'),
    ]
    for root in preferred:
        if (root / 'LoMa' / 'src' / 'loma').is_dir() and (root / 'gim' / 'networks' / 'lightglue').is_dir():
            return root
    if input_root.exists():
        for hit in sorted(input_root.glob('**/LoMa/src/loma')):
            root = hit.parents[2]
            if (root / 'gim' / 'networks' / 'lightglue').is_dir():
                return root
        wheel_hits = sorted(input_root.glob('**/pycolmap-*.whl'))
        if wheel_hits:
            return wheel_hits[0].parent
    return None

DEPS_DIR = existing_deps_root()
assert DEPS_DIR is not None and DEPS_DIR.exists(), (
    'Missing offline dependency files. Attach the Kaggle Dataset made from imc2024-final-project-deps.zip.'
)

wheel_hits = sorted(DEPS_DIR.glob('pycolmap-*.whl')) or sorted(DEPS_DIR.glob('**/pycolmap-*.whl'))
assert wheel_hits, f'Could not find pycolmap wheel under {DEPS_DIR}. Re-upload the newest imc2024-final-project-deps.zip.'
WHEEL_DIR = wheel_hits[0].parent
print('DEPS_DIR =', DEPS_DIR)
print('WHEEL_DIR =', WHEEL_DIR)

def find_dep(name):
    direct = DEPS_DIR / name
    if direct.exists():
        return direct
    hits = sorted(DEPS_DIR.glob(f'**/{name}'))
    return hits[0] if hits else direct

missing = [p for p in MODULES if not has_module(p)]
print('missing before offline install:', missing)
if missing:
    cmd = [
        sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
        '--find-links', str(WHEEL_DIR),
    ] + PIP_PACKAGES
    print('running offline install:', ' '.join(cmd))
    subprocess.check_call(cmd)

for p in MODULES:
    print(p, has_module(p))

remaining_missing = [p for p in MODULES if not has_module(p)]
assert not remaining_missing, (
    'Offline dependency install did not provide these modules: ' + repr(remaining_missing) +
    '. Re-upload the newest imc2024-final-project-deps.zip as a Kaggle Dataset and attach it.'
)

# Compatibility shim for Kaggle Python 3.12 / older checkpoint deserialization.
import torch
try:
    import torch._utils as _torch_private_utils
    setattr(torch, '_utils', _torch_private_utils)
    if not hasattr(torch, 'dtype'):
        setattr(torch, 'dtype', type(torch.float32))
    if not hasattr(torch, 'empty'):
        def _patched_empty(*size_args, **kwargs):
            if len(size_args) == 1:
                size = size_args[0]
            else:
                size = size_args
            if isinstance(size, int):
                shape = [size]
            else:
                try:
                    shape = list(size)
                except TypeError:
                    shape = [int(size)]
            return torch.ops.aten.empty.memory_format(
                shape,
                dtype=kwargs.get('dtype', torch.float32),
                layout=kwargs.get('layout', getattr(torch, 'strided', None)),
                device=kwargs.get('device', 'cpu'),
                pin_memory=bool(kwargs.get('pin_memory', False)),
                memory_format=kwargs.get('memory_format', getattr(torch, 'contiguous_format', None)),
            )
        setattr(torch, 'empty', _patched_empty)
    def _patched_element_size(dtype):
        try:
            return torch.empty((), dtype=dtype).element_size()
        except Exception:
            return {
                'torch.bool': 1, 'torch.uint8': 1, 'torch.int8': 1,
                'torch.float16': 2, 'torch.bfloat16': 2,
                'torch.int32': 4, 'torch.float32': 4,
                'torch.int64': 8, 'torch.float64': 8,
                'torch.complex64': 8, 'torch.complex128': 16,
            }.get(str(dtype), 4)
    _torch_private_utils._element_size = _patched_element_size
    print('torch serialization shim: ok', 'has_dtype=', hasattr(torch, 'dtype'), 'has_empty=', hasattr(torch, 'empty'))
except Exception as exc:
    print('torch serialization shim warning:', repr(exc))

cache_dir = Path('/tmp/xdg-cache/torch/hub/checkpoints')
cache_dir.mkdir(parents=True, exist_ok=True)
weight_map = {
    'aliked-n16.pth': 'aliked-n16.pth',
    'aliked_lightglue.pth': 'aliked_lightglue_v0-1_arxiv.pth',
    'loma_B.pt': 'loma_B.pt',
    'dad.pth': 'dad.pth',
    'dedode_descriptor_G.pth': 'dedode_descriptor_G.pth',
    'dinov2_vitl14_pretrain.pth': 'dinov2_vitl14_pretrain.pth',
}
required_cache_files = []
for src_name, dst_name in weight_map.items():
    src_path = find_dep(src_name)
    dst_path = cache_dir / dst_name
    if not src_path.exists():
        raise FileNotFoundError(f'Missing required weight in deps: {src_name}')
    if (not dst_path.exists()) or dst_path.stat().st_size != src_path.stat().st_size:
        shutil.copy(src_path, dst_path)
        print('copied weight', src_path, '->', dst_path)
    required_cache_files.append(dst_path)

missing_cache = [str(p) for p in required_cache_files if not p.exists() or p.stat().st_size == 0]
assert not missing_cache, 'Missing required torch checkpoint cache files: ' + repr(missing_cache)

LOMA_ROOT = find_dep('LoMa')
assert (LOMA_ROOT / 'src' / 'loma').is_dir(), 'Missing LoMa repo in deps: ' + str(LOMA_ROOT)
print('LOMA_ROOT =', LOMA_ROOT)

GIM_ROOT = find_dep('gim')
GIM_WEIGHTS = find_dep('gim_lightglue_100h.ckpt')
if not GIM_WEIGHTS.exists() and (GIM_ROOT / 'weights' / 'gim_lightglue_100h.ckpt').exists():
    GIM_WEIGHTS = GIM_ROOT / 'weights' / 'gim_lightglue_100h.ckpt'
assert (GIM_ROOT / 'networks' / 'lightglue').is_dir(), 'Missing GIM-LightGlue code in deps: ' + str(GIM_ROOT)
assert GIM_WEIGHTS.exists() and GIM_WEIGHTS.stat().st_size > 40_000_000, 'Missing or incomplete GIM-LightGlue checkpoint: ' + str(GIM_WEIGHTS)
print('GIM_ROOT =', GIM_ROOT)
print('GIM_WEIGHTS =', GIM_WEIGHTS, GIM_WEIGHTS.stat().st_size)


In [ ]:
import sys, types
module = types.ModuleType('top_imc_pipeline_embedded')
sys.modules[module.__name__] = module
code = '"""Top-style IMC 2024 pipeline: ALIKED + LightGlue + COLMAP + transparent prior.\n\nThis is the non-oracle pipeline intended for a real test split.  It follows the\npublic high-ranking IMC 2024 recipe:\n\n1. exhaustive pair proposal for small scenes;\n2. ALIKED sparse keypoints at high resolution;\n3. LightGlue matching with 0/90/180/270 degree rotation search;\n4. MAGSAC fundamental-matrix verification;\n5. optional DBSCAN crop rematching for dense high-confidence areas;\n6. pycolmap incremental SfM;\n7. transparent scenes use a match-count ordering and a circular camera prior.\n\nRun it with the `project` conda environment:\n\n    conda run -n project python src/top_imc_pipeline.py --data_root data\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport itertools\nimport math\nimport os\nimport random\nimport shutil\nimport sys\nimport warnings\nfrom contextlib import redirect_stdout\nfrom dataclasses import dataclass, field, replace\nfrom io import StringIO\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport cv2\nimport numpy as np\nimport pycolmap\nimport torch\nfrom PIL import Image, ImageOps\nfrom sklearn.cluster import DBSCAN\nfrom tqdm import tqdm\n\nfrom lightglue import ALIKED, DISK, DoGHardNet, LightGlue, SIFT, SuperPoint\nfrom lightglue.utils import load_image, rbd\n\n\nos.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")\nos.environ.setdefault("XDG_CACHE_HOME", "/tmp/xdg-cache")\nos.environ.setdefault("PYTHONHASHSEED", "0")\nwarnings.filterwarnings(\n    "ignore",\n    message=r"`torch\\.cuda\\.amp\\.autocast\\(args\\.\\.\\.\\)` is deprecated.*",\n    category=FutureWarning,\n)\nwarnings.filterwarnings(\n    "ignore",\n    message=r".*torch\\.cuda\\.amp\\.custom_fwd.*",\n    category=FutureWarning,\n)\nwarnings.filterwarnings(\n    "ignore",\n    message=r"Unable to find acceptable character detection dependency.*",\n)\n\n\ndef patch_torch_serialization() -> None:\n    try:\n        import torch._utils as torch_private_utils\n\n        setattr(torch, "_utils", torch_private_utils)\n        if not hasattr(torch, "dtype"):\n            setattr(torch, "dtype", type(torch.float32))\n        if not hasattr(torch, "empty"):\n\n            def patched_empty(*size_args: object, **kwargs: object) -> torch.Tensor:\n                if len(size_args) == 1:\n                    size = size_args[0]\n                else:\n                    size = size_args\n                if isinstance(size, int):\n                    shape = [size]\n                else:\n                    try:\n                        shape = list(size)  # type: ignore[arg-type]\n                    except TypeError:\n                        shape = [int(size)]  # type: ignore[arg-type]\n                return torch.ops.aten.empty.memory_format(\n                    shape,\n                    dtype=kwargs.get("dtype", torch.float32),\n                    layout=kwargs.get("layout", getattr(torch, "strided", None)),\n                    device=kwargs.get("device", "cpu"),\n                    pin_memory=bool(kwargs.get("pin_memory", False)),\n                    memory_format=kwargs.get(\n                        "memory_format", getattr(torch, "contiguous_format", None)\n                    ),\n                )\n\n            setattr(torch, "empty", patched_empty)\n\n        def element_size(dtype: object) -> int:\n            try:\n                return torch.empty((), dtype=dtype).element_size()\n            except Exception:\n                sizes = {\n                    "torch.bool": 1,\n                    "torch.uint8": 1,\n                    "torch.int8": 1,\n                    "torch.float8_e4m3fn": 1,\n                    "torch.float8_e5m2": 1,\n                    "torch.int16": 2,\n                    "torch.float16": 2,\n                    "torch.bfloat16": 2,\n                    "torch.int32": 4,\n                    "torch.float32": 4,\n                    "torch.int64": 8,\n                    "torch.float64": 8,\n                    "torch.complex64": 8,\n                    "torch.complex128": 16,\n                }\n                return sizes.get(str(dtype), 4)\n\n        torch_private_utils._element_size = element_size\n    except Exception:\n        pass\n\n\npatch_torch_serialization()\n\n\nIMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}\n\n\n@dataclass(frozen=True)\nclass SubmissionRow:\n    image_path: str\n    dataset: str\n    scene: str\n\n    @property\n    def image_name(self) -> str:\n        return Path(self.image_path).name\n\n\n@dataclass\nclass RuntimeConfig:\n    data_root: Path\n    output: Path\n    work_root: Path\n    report: Path\n    scene_filter: Optional[str] = None\n    seed: int = 3407\n    device: str = "auto"\n    max_images: int = 0\n    max_pairs: int = 0\n    max_keypoints: int = 4096\n    resize: int = 1600\n    detection_threshold: float = 0.01\n    feature_backend: str = "aliked"\n    matcher_backend: str = "lightglue"\n    min_matches: int = 80\n    ransac_threshold: float = 0.75\n    ransac_confidence: float = 0.9999\n    ransac_max_iters: int = 20000\n    rotate_search: bool = True\n    crop_rematch: bool = True\n    crop_resize: int = 2400\n    crop_max_keypoints: int = 2048\n    crop_min_seed_matches: int = 160\n    crop_dbscan_eps: float = 96.0\n    crop_dbscan_min_samples: int = 12\n    crop_padding: int = 192\n    roma_fallback: bool = False\n    roma_max_matches: int = 5000\n    roma_fallback_max_pairs: int = 0\n    edm_fallback: bool = False\n    edm_fallback_max_pairs: int = 0\n    edm_root: Path = Path("vendor/EDM")\n    edm_width: int = 640\n    edm_height: int = 480\n    mast3r_root: Path = Path("vendor/MASt3R")\n    mast3r_weights: Path = Path("kaggle_deps/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth")\n    mast3r_image_size: int = 512\n    mast3r_subsample: int = 8\n    mast3r_max_matches: int = 5000\n    mast3r_min_tri_angle: float = 2.0\n    mast3r_relaxed_min_tri_angle: float = 0.25\n    mast3r_relaxed_max_pairs: int = 200\n    mast3r_fallback: bool = False\n    mast3r_fallback_max_pairs: int = 120\n    loma_root: Path = Path("vendor/LoMa")\n    loma_variant: str = "B"\n    loma_num_keypoints: int = 4096\n    loma_filter_threshold: float = 0.1\n    gim_root: Path = Path("vendor/gim")\n    gim_weights: Path = Path("vendor/gim/weights/gim_lightglue_100h.ckpt")\n    gim_max_keypoints: int = 2048\n    gim_resize: int = 1920\n    gim_filter_threshold: float = 0.1\n    scene_fallback_min_ratio: float = 0.0\n    scene_fallback_backend: str = "aliked_lightglue"\n    transparent_use_sfm: bool = False\n    camera_model: str = "SIMPLE_PINHOLE"\n    focal_prior: float = 1.2\n    mapper_min_model_size: int = 3\n    mapper_max_num_models: int = 3\n    oracle_lookup: bool = False\n\n\n@dataclass\nclass ImageFeatures:\n    path: Path\n    width: int\n    height: int\n    variants: Dict[int, Dict[str, torch.Tensor]] = field(default_factory=dict)\n    global_keypoints: List[np.ndarray] = field(default_factory=list)\n\n\n@dataclass\nclass PairMatch:\n    name0: str\n    name1: str\n    idx0: np.ndarray\n    idx1: np.ndarray\n    kpts0: np.ndarray\n    kpts1: np.ndarray\n    rotation_k1: int\n    raw_matches: int\n    verified_matches: int\n    source: str\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef choose_device(value: str) -> torch.device:\n    if value != "auto":\n        return torch.device(value)\n    return torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef read_submission_template(path: Path) -> List[SubmissionRow]:\n    with path.open("r", newline="", encoding="utf-8") as handle:\n        reader = csv.DictReader(handle)\n        return [\n            SubmissionRow(row["image_path"], row["dataset"], row["scene"])\n            for row in reader\n        ]\n\n\ndef read_categories(path: Path) -> Dict[str, str]:\n    if not path.is_file():\n        return {}\n    with path.open("r", newline="", encoding="utf-8") as handle:\n        return {row["scene"]: row["categories"] for row in csv.DictReader(handle)}\n\n\ndef parse_vector(text: str, size: int) -> np.ndarray:\n    values = [float(x) for x in text.split(";") if x != ""]\n    if len(values) != size:\n        raise ValueError(f"Expected {size} values, got {len(values)} in {text!r}")\n    return np.asarray(values, dtype=np.float64)\n\n\ndef load_train_poses(data_root: Path) -> Dict[Tuple[str, str, str], Tuple[np.ndarray, np.ndarray]]:\n    path = data_root / "train" / "train_labels.csv"\n    if not path.is_file():\n        return {}\n    out: Dict[Tuple[str, str, str], Tuple[np.ndarray, np.ndarray]] = {}\n    with path.open("r", newline="", encoding="utf-8") as handle:\n        for row in csv.DictReader(handle):\n            out[(row["dataset"], row["scene"], row["image_name"])] = (\n                parse_vector(row["rotation_matrix"], 9).reshape(3, 3),\n                parse_vector(row["translation_vector"], 3),\n            )\n    return out\n\n\ndef encode_matrix(matrix: np.ndarray) -> str:\n    return ";".join(f"{float(x):.17g}" for x in matrix.reshape(-1))\n\n\ndef encode_vector(vector: np.ndarray) -> str:\n    return ";".join(f"{float(x):.17g}" for x in vector.reshape(-1))\n\n\ndef group_rows(rows: Iterable[SubmissionRow]) -> Dict[Tuple[str, str], List[SubmissionRow]]:\n    groups: Dict[Tuple[str, str], List[SubmissionRow]] = {}\n    for row in rows:\n        groups.setdefault((row.dataset, row.scene), []).append(row)\n    return groups\n\n\ndef list_scene_images(data_root: Path, rows: Sequence[SubmissionRow], max_images: int) -> List[Path]:\n    if not rows:\n        return []\n    scene_dir = data_root / "test" / rows[0].scene / "images"\n    requested = [data_root / row.image_path for row in rows]\n    requested = [path for path in requested if path.is_file()]\n    if max_images and len(requested) > max_images:\n        requested = requested[:max_images]\n    all_images = sorted(\n        [path for path in scene_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS]\n    )\n    if max_images and len(all_images) > max_images:\n        extras = [path for path in all_images if path not in set(requested)]\n        all_images = requested + extras[: max(0, max_images - len(requested))]\n    return all_images\n\n\ndef image_size(path: Path) -> Tuple[int, int]:\n    with Image.open(path) as image:\n        return image.size\n\n\ndef rot90_tensor(image: torch.Tensor, k: int) -> torch.Tensor:\n    if k % 4 == 0:\n        return image\n    return torch.rot90(image, k=k, dims=(-2, -1))\n\n\ndef unrotate_keypoints(keypoints: np.ndarray, width: int, height: int, k: int) -> np.ndarray:\n    if k % 4 == 0:\n        return keypoints.copy()\n    x = keypoints[:, 0].copy()\n    y = keypoints[:, 1].copy()\n    if k % 4 == 1:  # tensor was rotated 90 degrees counter-clockwise\n        return np.stack([width - 1.0 - y, x], axis=1)\n    if k % 4 == 2:\n        return np.stack([width - 1.0 - x, height - 1.0 - y], axis=1)\n    return np.stack([y, height - 1.0 - x], axis=1)\n\n\ndef clamp_keypoints(keypoints: np.ndarray, width: int, height: int) -> np.ndarray:\n    keypoints = keypoints.copy()\n    keypoints[:, 0] = np.clip(keypoints[:, 0], 0.0, width - 1.0)\n    keypoints[:, 1] = np.clip(keypoints[:, 1], 0.0, height - 1.0)\n    return keypoints\n\n\ndef append_keypoints(store: ImageFeatures, keypoints: np.ndarray) -> np.ndarray:\n    if keypoints.size == 0:\n        return np.zeros((0,), dtype=np.uint32)\n    keypoints = clamp_keypoints(keypoints.astype(np.float32), store.width, store.height)\n    offset = sum(len(chunk) for chunk in store.global_keypoints)\n    store.global_keypoints.append(keypoints)\n    return np.arange(offset, offset + len(keypoints), dtype=np.uint32)\n\n\ndef get_global_keypoints(store: ImageFeatures) -> np.ndarray:\n    if not store.global_keypoints:\n        return np.zeros((0, 2), dtype=np.float32)\n    return np.concatenate(store.global_keypoints, axis=0).astype(np.float32)\n\n\ndef create_extractor(\n    backend: str,\n    max_keypoints: int,\n    detection_threshold: float,\n    resize: int,\n    device: torch.device,\n) -> torch.nn.Module:\n    backend = backend.lower()\n    conf = {\n        "max_num_keypoints": max_keypoints,\n        "detection_threshold": detection_threshold,\n        "resize": resize,\n    }\n    if backend == "aliked":\n        extractor = ALIKED(**conf)\n    elif backend == "disk":\n        extractor = DISK(**conf)\n    elif backend == "superpoint":\n        extractor = SuperPoint(**conf)\n    elif backend == "sift":\n        extractor = SIFT(**conf)\n    elif backend == "doghardnet":\n        extractor = DoGHardNet(**conf)\n    else:\n        raise ValueError(f"Unsupported feature backend: {backend}")\n    return extractor.eval().to(device)\n\n\ndef create_matcher(backend: str, device: torch.device) -> LightGlue:\n    return LightGlue(features=backend.lower(), depth_confidence=-1, width_confidence=-1).eval().to(device)\n\n\ndef extract_variant(\n    extractor: torch.nn.Module,\n    image_tensor: torch.Tensor,\n    width: int,\n    height: int,\n    k: int,\n    device: torch.device,\n) -> Dict[str, torch.Tensor]:\n    image = rot90_tensor(image_tensor, k).to(device)\n    with torch.inference_mode():\n        features = extractor.extract(image)\n    keypoints = rbd(features)["keypoints"].detach().cpu().numpy()\n    keypoints = unrotate_keypoints(keypoints, width, height, k)\n    keypoints = clamp_keypoints(keypoints, width, height)\n    features["keypoints_orig"] = torch.from_numpy(keypoints).to(device)[None]\n    return features\n\n\ndef extract_scene_features(\n    paths: Sequence[Path],\n    cfg: RuntimeConfig,\n    extractor: torch.nn.Module,\n    device: torch.device,\n) -> Dict[str, ImageFeatures]:\n    features: Dict[str, ImageFeatures] = {}\n    rotations = [0, 1, 2, 3] if cfg.rotate_search else [0]\n    for path in tqdm(paths, desc=f"{cfg.feature_backend} features", dynamic_ncols=True):\n        width, height = image_size(path)\n        store = ImageFeatures(path=path, width=width, height=height)\n        image = load_image(path).to(device)\n        for k in rotations:\n            store.variants[k] = extract_variant(extractor, image, width, height, k, device)\n        base_kpts = rbd({"keypoints_orig": store.variants[0]["keypoints_orig"]})[\n            "keypoints_orig"\n        ].detach().cpu().numpy()\n        append_keypoints(store, base_kpts)\n        features[path.name] = store\n    return features\n\n\ndef init_scene_stores(paths: Sequence[Path]) -> Dict[str, ImageFeatures]:\n    stores: Dict[str, ImageFeatures] = {}\n    for path in paths:\n        width, height = image_size(path)\n        stores[path.name] = ImageFeatures(path=path, width=width, height=height)\n    return stores\n\n\ndef lightglue_match(\n    matcher: LightGlue,\n    feat0: Dict[str, torch.Tensor],\n    feat1: Dict[str, torch.Tensor],\n) -> np.ndarray:\n    with torch.inference_mode():\n        pred = rbd(matcher({"image0": feat0, "image1": feat1}))\n    matches = pred.get("matches")\n    if matches is None:\n        return np.zeros((0, 2), dtype=np.int64)\n    return matches.detach().cpu().numpy().astype(np.int64)\n\n\ndef fundamental_inliers(\n    kpts0: np.ndarray,\n    kpts1: np.ndarray,\n    cfg: RuntimeConfig,\n) -> Tuple[np.ndarray, Optional[np.ndarray]]:\n    if len(kpts0) < 8:\n        return np.zeros((len(kpts0),), dtype=bool), None\n    F, inliers = cv2.findFundamentalMat(\n        kpts0.astype(np.float32),\n        kpts1.astype(np.float32),\n        cv2.USAC_MAGSAC,\n        cfg.ransac_threshold,\n        cfg.ransac_confidence,\n        cfg.ransac_max_iters,\n    )\n    if F is None or inliers is None:\n        return np.zeros((len(kpts0),), dtype=bool), None\n    return inliers.reshape(-1).astype(bool), F.astype(np.float64)\n\n\ndef match_one_pair(\n    name0: str,\n    name1: str,\n    stores: Dict[str, ImageFeatures],\n    matcher: LightGlue,\n    cfg: RuntimeConfig,\n) -> Tuple[Optional[PairMatch], Optional[np.ndarray]]:\n    store0 = stores[name0]\n    store1 = stores[name1]\n    feat0 = store0.variants[0]\n    best: Tuple[int, np.ndarray] = (0, np.zeros((0, 2), dtype=np.int64))\n\n    for k, feat1 in store1.variants.items():\n        matches = lightglue_match(matcher, feat0, feat1)\n        if len(matches) > len(best[1]):\n            best = (k, matches)\n\n    rotation_k1, matches = best\n    if len(matches) < cfg.min_matches:\n        return None, None\n\n    kpts0_all = rbd({"keypoints_orig": feat0["keypoints_orig"]})["keypoints_orig"].detach().cpu().numpy()\n    kpts1_all = rbd({"keypoints_orig": store1.variants[rotation_k1]["keypoints_orig"]})[\n        "keypoints_orig"\n    ].detach().cpu().numpy()\n    mkpts0 = kpts0_all[matches[:, 0]]\n    mkpts1 = kpts1_all[matches[:, 1]]\n    inliers, F = fundamental_inliers(mkpts0, mkpts1, cfg)\n    if int(inliers.sum()) < cfg.min_matches:\n        return None, None\n\n    matches = matches[inliers]\n    mkpts0 = mkpts0[inliers]\n    mkpts1 = mkpts1[inliers]\n\n    if rotation_k1 == 0:\n        idx0 = matches[:, 0].astype(np.uint32)\n        idx1 = matches[:, 1].astype(np.uint32)\n    else:\n        idx0 = matches[:, 0].astype(np.uint32)\n        idx1 = append_keypoints(store1, mkpts1)\n\n    pair = PairMatch(\n        name0=name0,\n        name1=name1,\n        idx0=idx0,\n        idx1=idx1,\n        kpts0=mkpts0.astype(np.float32),\n        kpts1=mkpts1.astype(np.float32),\n        rotation_k1=rotation_k1,\n        raw_matches=len(best[1]),\n        verified_matches=len(matches),\n        source="aliked_lightglue",\n    )\n    return pair, F\n\n\ndef crop_box(points: np.ndarray, width: int, height: int, padding: int) -> Tuple[int, int, int, int]:\n    x0 = max(0, int(np.floor(points[:, 0].min())) - padding)\n    y0 = max(0, int(np.floor(points[:, 1].min())) - padding)\n    x1 = min(width, int(np.ceil(points[:, 0].max())) + padding)\n    y1 = min(height, int(np.ceil(points[:, 1].max())) + padding)\n    if x1 <= x0 or y1 <= y0:\n        return 0, 0, width, height\n    return x0, y0, x1, y1\n\n\ndef extract_crop_features(\n    extractor: torch.nn.Module,\n    image: np.ndarray,\n    box: Tuple[int, int, int, int],\n    device: torch.device,\n) -> Dict[str, torch.Tensor]:\n    x0, y0, x1, y1 = box\n    crop = image[y0:y1, x0:x1]\n    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)\n    tensor = torch.from_numpy(crop_rgb).float().permute(2, 0, 1) / 255.0\n    tensor = tensor.to(device)\n    with torch.inference_mode():\n        features = extractor.extract(tensor)\n    keypoints = rbd(features)["keypoints"].detach().cpu().numpy()\n    keypoints[:, 0] += x0\n    keypoints[:, 1] += y0\n    features["keypoints_orig"] = torch.from_numpy(keypoints.astype(np.float32)).to(device)[None]\n    return features\n\n\ndef crop_rematch_pair(\n    pair: PairMatch,\n    stores: Dict[str, ImageFeatures],\n    extractor: ALIKED,\n    matcher: LightGlue,\n    cfg: RuntimeConfig,\n    device: torch.device,\n) -> Optional[PairMatch]:\n    if len(pair.kpts0) < cfg.crop_min_seed_matches:\n        return None\n\n    midpoint = (pair.kpts0 + pair.kpts1) * 0.5\n    labels = DBSCAN(eps=cfg.crop_dbscan_eps, min_samples=cfg.crop_dbscan_min_samples).fit_predict(midpoint)\n    valid_labels = [label for label in sorted(set(labels)) if label >= 0]\n    if not valid_labels:\n        return None\n\n    store0 = stores[pair.name0]\n    store1 = stores[pair.name1]\n    image0 = cv2.imread(str(store0.path))\n    image1 = cv2.imread(str(store1.path))\n    if image0 is None or image1 is None:\n        return None\n\n    all_idx0: List[np.ndarray] = []\n    all_idx1: List[np.ndarray] = []\n    all_kpts0: List[np.ndarray] = []\n    all_kpts1: List[np.ndarray] = []\n\n    for label in valid_labels[:4]:\n        mask = labels == label\n        if int(mask.sum()) < cfg.crop_dbscan_min_samples:\n            continue\n        box0 = crop_box(pair.kpts0[mask], store0.width, store0.height, cfg.crop_padding)\n        box1 = crop_box(pair.kpts1[mask], store1.width, store1.height, cfg.crop_padding)\n        feat0 = extract_crop_features(extractor, image0, box0, device)\n        feat1 = extract_crop_features(extractor, image1, box1, device)\n        matches = lightglue_match(matcher, feat0, feat1)\n        if len(matches) < cfg.min_matches:\n            continue\n        kpts0_all = rbd({"keypoints_orig": feat0["keypoints_orig"]})["keypoints_orig"].detach().cpu().numpy()\n        kpts1_all = rbd({"keypoints_orig": feat1["keypoints_orig"]})["keypoints_orig"].detach().cpu().numpy()\n        kpts0 = kpts0_all[matches[:, 0]]\n        kpts1 = kpts1_all[matches[:, 1]]\n        inliers, _ = fundamental_inliers(kpts0, kpts1, cfg)\n        if int(inliers.sum()) < cfg.min_matches:\n            continue\n        kpts0 = kpts0[inliers].astype(np.float32)\n        kpts1 = kpts1[inliers].astype(np.float32)\n        all_idx0.append(append_keypoints(store0, kpts0))\n        all_idx1.append(append_keypoints(store1, kpts1))\n        all_kpts0.append(kpts0)\n        all_kpts1.append(kpts1)\n\n    if not all_idx0:\n        return None\n\n    return PairMatch(\n        name0=pair.name0,\n        name1=pair.name1,\n        idx0=np.concatenate(all_idx0).astype(np.uint32),\n        idx1=np.concatenate(all_idx1).astype(np.uint32),\n        kpts0=np.concatenate(all_kpts0).astype(np.float32),\n        kpts1=np.concatenate(all_kpts1).astype(np.float32),\n        rotation_k1=pair.rotation_k1,\n        raw_matches=sum(len(x) for x in all_idx0),\n        verified_matches=sum(len(x) for x in all_idx0),\n        source="dbscan_crop_rematch",\n    )\n\n\ndef roma_match_pair(\n    name0: str,\n    name1: str,\n    stores: Dict[str, ImageFeatures],\n    roma_model: object,\n    cfg: RuntimeConfig,\n) -> Tuple[Optional[PairMatch], Optional[np.ndarray]]:\n    store0 = stores[name0]\n    store1 = stores[name1]\n    try:\n        with torch.inference_mode():\n            warp, certainty = roma_model.match(str(store0.path), str(store1.path))\n            matches, certainty = roma_model.sample(warp, certainty, num=cfg.roma_max_matches)\n            kpts0_t, kpts1_t = roma_model.to_pixel_coordinates(\n                matches,\n                store0.height,\n                store0.width,\n                store1.height,\n                store1.width,\n            )\n    except Exception as exc:\n        print(f"RoMa fallback failed for {name0} / {name1}: {exc}")\n        return None, None\n\n    kpts0 = kpts0_t.detach().cpu().numpy().astype(np.float32)\n    kpts1 = kpts1_t.detach().cpu().numpy().astype(np.float32)\n    if len(kpts0) < cfg.min_matches:\n        return None, None\n    inliers, F = fundamental_inliers(kpts0, kpts1, cfg)\n    if int(inliers.sum()) < cfg.min_matches:\n        return None, None\n\n    kpts0 = kpts0[inliers]\n    kpts1 = kpts1[inliers]\n    idx0 = append_keypoints(store0, kpts0)\n    idx1 = append_keypoints(store1, kpts1)\n    pair = PairMatch(\n        name0=name0,\n        name1=name1,\n        idx0=idx0,\n        idx1=idx1,\n        kpts0=kpts0.astype(np.float32),\n        kpts1=kpts1.astype(np.float32),\n        rotation_k1=0,\n        raw_matches=int(len(matches)),\n        verified_matches=int(len(kpts0)),\n        source="tiny_roma_fallback",\n    )\n    return pair, F\n\n\ndef create_tiny_roma_model(device: torch.device) -> torch.nn.Module:\n    from romatch import tiny_roma_v1_outdoor\n\n    hub_root = Path(os.environ.get("XDG_CACHE_HOME", "~/.cache")).expanduser() / "torch" / "hub"\n    checkpoint_dir = hub_root / "checkpoints"\n\n    tiny_roma_path = checkpoint_dir / "tiny_roma_v1_outdoor.pth"\n    if not tiny_roma_path.is_file():\n        raise FileNotFoundError(\n            f"Missing offline Tiny RoMa weights: {tiny_roma_path}. "\n            "The notebook bootstrap must copy tiny_roma_v1_outdoor.pth from the deps dataset."\n        )\n    weights = torch.load(tiny_roma_path, map_location=device)\n\n    xfeat_repo = hub_root / "verlab_accelerated_features_main"\n    xfeat_path = checkpoint_dir / "xfeat.pt"\n    if not xfeat_repo.is_dir() or not (xfeat_repo / "hubconf.py").is_file():\n        raise FileNotFoundError(\n            f"Missing offline XFeat torch-hub repo: {xfeat_repo}. "\n            "The notebook bootstrap must copy verlab_accelerated_features_main from the deps dataset."\n        )\n    if not xfeat_path.is_file():\n        raise FileNotFoundError(\n            f"Missing offline XFeat weights: {xfeat_path}. "\n            "The notebook bootstrap must copy xfeat.pt from the deps dataset."\n        )\n    sys.path.insert(0, str(xfeat_repo))\n    from modules.xfeat import XFeat\n\n    xfeat = XFeat(str(xfeat_path), top_k=4096).net\n\n    return tiny_roma_v1_outdoor(device=str(device), weights=weights, xfeat=xfeat)\n\n\ndef create_edm_model(cfg: RuntimeConfig, device: torch.device) -> torch.nn.Module:\n    edm_root = cfg.edm_root.resolve()\n    sys.path.insert(0, str(edm_root))\n    existing_src = sys.modules.get("src")\n    existing_src_file = str(getattr(existing_src, "__file__", "")) if existing_src else ""\n    if existing_src is not None and not existing_src_file.startswith(str(edm_root)):\n        for name in list(sys.modules):\n            if name == "src" or name.startswith("src."):\n                del sys.modules[name]\n    from src.config.default import get_cfg_defaults\n    from src.edm import EDM\n    from src.utils.misc import lower_config\n\n    config = get_cfg_defaults()\n    config.merge_from_file(str(edm_root / "configs/edm/outdoor/edm_base.py"))\n    config.merge_from_file(str(edm_root / "configs/data/megadepth_test_1500.py"))\n    config.EDM.COARSE.MCONF_THR = 0.2\n    config.EDM.COARSE.BORDER_RM = 2\n    config.EDM.TEST_RES_H = cfg.edm_height\n    config.EDM.TEST_RES_W = cfg.edm_width\n    model = EDM(config=lower_config(config)["edm"]).to(device)\n    state = torch.load(edm_root / "weights/edm_outdoor.ckpt", map_location=device)\n    model.load_state_dict(state["state_dict"])\n    return model.eval()\n\n\ndef create_loma_model(cfg: RuntimeConfig, device: torch.device) -> torch.nn.Module:\n    loma_src = cfg.loma_root.resolve() / "src"\n    if not (loma_src / "loma").is_dir():\n        raise FileNotFoundError(f"Missing LoMa source repo: {loma_src.parent}")\n    if str(loma_src) not in sys.path:\n        sys.path.insert(0, str(loma_src))\n\n    from loma import LoMa, LoMaB, LoMaB128, LoMaG, LoMaL, LoMaR\n\n    variants = {\n        "B128": LoMaB128,\n        "B": LoMaB,\n        "L": LoMaL,\n        "G": LoMaG,\n        "R": LoMaR,\n    }\n    variant = cfg.loma_variant.upper()\n    if variant not in variants:\n        raise ValueError(f"Unsupported LoMa variant: {cfg.loma_variant}")\n    model_cfg = variants[variant](\n        num_keypoints=cfg.loma_num_keypoints,\n        filter_threshold=cfg.loma_filter_threshold,\n        compile=False,\n    )\n    return LoMa(model_cfg).eval().to(device)\n\n\ndef extract_loma_scene_features(\n    paths: Sequence[Path],\n    cfg: RuntimeConfig,\n    loma_model: torch.nn.Module,\n    device: torch.device,\n) -> Dict[str, ImageFeatures]:\n    from loma.loma import to_pixel_coords\n\n    features: Dict[str, ImageFeatures] = {}\n    for path in tqdm(paths, desc=f"LoMa-{cfg.loma_variant} features", dynamic_ncols=True):\n        width, height = image_size(path)\n        store = ImageFeatures(path=path, width=width, height=height)\n        with torch.inference_mode():\n            keypoints, descriptors, h, w = loma_model.detect_and_describe(\n                str(path), num_keypoints=cfg.loma_num_keypoints\n            )\n        pixel_keypoints = to_pixel_coords(keypoints, h, w).detach().cpu().numpy()[0].astype(np.float32)\n        pixel_keypoints = clamp_keypoints(pixel_keypoints, width, height)\n        append_keypoints(store, pixel_keypoints)\n        store.variants[0] = {\n            "keypoints": keypoints.detach().to(device),\n            "descriptors": descriptors.detach().to(device),\n            "keypoints_orig": torch.from_numpy(pixel_keypoints).to(device)[None],\n        }\n        features[path.name] = store\n    return features\n\n\ndef loma_match_pair(\n    name0: str,\n    name1: str,\n    stores: Dict[str, ImageFeatures],\n    loma_model: torch.nn.Module,\n    cfg: RuntimeConfig,\n) -> Tuple[Optional[PairMatch], Optional[np.ndarray]]:\n    from loma.loma import filter_matches\n\n    store0 = stores[name0]\n    store1 = stores[name1]\n    feat0 = store0.variants[0]\n    feat1 = store1.variants[0]\n    with torch.inference_mode():\n        scores = loma_model(\n            feat0["keypoints"],\n            feat1["keypoints"],\n            feat0["descriptors"],\n            feat1["descriptors"],\n        )["scores"]\n        m0, _, _, _ = filter_matches(scores, cfg.loma_filter_threshold)\n    valid = m0[0] > -1\n    idx0_t = torch.where(valid)[0]\n    idx1_t = m0[0][valid]\n    if int(valid.sum()) < cfg.min_matches:\n        return None, None\n\n    idx0 = idx0_t.detach().cpu().numpy().astype(np.uint32)\n    idx1 = idx1_t.detach().cpu().numpy().astype(np.uint32)\n    kpts0_all = feat0["keypoints_orig"][0].detach().cpu().numpy()\n    kpts1_all = feat1["keypoints_orig"][0].detach().cpu().numpy()\n    kpts0 = kpts0_all[idx0]\n    kpts1 = kpts1_all[idx1]\n    inliers, F = fundamental_inliers(kpts0, kpts1, cfg)\n    if int(inliers.sum()) < cfg.min_matches:\n        return None, None\n\n    idx0 = idx0[inliers]\n    idx1 = idx1[inliers]\n    kpts0 = kpts0[inliers]\n    kpts1 = kpts1[inliers]\n    return (\n        PairMatch(\n            name0=name0,\n            name1=name1,\n            idx0=idx0,\n            idx1=idx1,\n            kpts0=kpts0.astype(np.float32),\n            kpts1=kpts1.astype(np.float32),\n            rotation_k1=0,\n            raw_matches=int(valid.sum()),\n            verified_matches=int(len(kpts0)),\n            source=f"loma_{cfg.loma_variant.lower()}",\n        ),\n        F,\n    )\n\n\ndef create_gim_lightglue_models(\n    cfg: RuntimeConfig,\n    device: torch.device,\n) -> Tuple[torch.nn.Module, torch.nn.Module]:\n    gim_root = cfg.gim_root.resolve()\n    weights_path = cfg.gim_weights\n    if not weights_path.is_absolute():\n        weights_path = (Path.cwd() / weights_path).resolve()\n    if not (gim_root / "networks" / "lightglue").is_dir():\n        raise FileNotFoundError(f"Missing GIM repo: {gim_root}")\n    if not weights_path.is_file():\n        raise FileNotFoundError(f"Missing GIM-LightGlue checkpoint: {weights_path}")\n    if str(gim_root) not in sys.path:\n        sys.path.insert(0, str(gim_root))\n\n    from networks.lightglue.superpoint import SuperPoint as GimSuperPoint\n    from networks.lightglue.models.matchers.lightglue import LightGlue as GimLightGlue\n\n    state = torch.load(weights_path, map_location="cpu")\n    state_dict = state["state_dict"] if "state_dict" in state else state\n\n    detector = GimSuperPoint(\n        {\n            "max_num_keypoints": cfg.gim_max_keypoints,\n            "force_num_keypoints": True,\n            "detection_threshold": 0.0,\n            "nms_radius": 3,\n            "trainable": False,\n        }\n    )\n    detector_state = dict(state_dict)\n    for key in list(detector_state.keys()):\n        if key.startswith("model."):\n            detector_state.pop(key)\n        elif key.startswith("superpoint."):\n            detector_state[key.replace("superpoint.", "", 1)] = detector_state.pop(key)\n    detector.load_state_dict(detector_state)\n\n    matcher = GimLightGlue(\n        {\n            "filter_threshold": cfg.gim_filter_threshold,\n            "flash": False,\n            "checkpointed": False,\n            "depth_confidence": -1,\n            "width_confidence": -1,\n        }\n    )\n    matcher_state = dict(state_dict)\n    for key in list(matcher_state.keys()):\n        if key.startswith("superpoint."):\n            matcher_state.pop(key)\n        elif key.startswith("model."):\n            matcher_state[key.replace("model.", "", 1)] = matcher_state.pop(key)\n    matcher.load_state_dict(matcher_state)\n    return detector.eval().to(device), matcher.eval().to(device)\n\n\ndef gim_load_tensor(\n    path: Path,\n    max_size: int,\n    rotation_k: int,\n    device: torch.device,\n) -> Tuple[torch.Tensor, Tuple[int, int], np.ndarray]:\n    image = ImageOps.exif_transpose(Image.open(path)).convert("L")\n    orig_w, orig_h = image.size\n    array = np.asarray(image)\n    if rotation_k:\n        array = np.ascontiguousarray(np.rot90(array, rotation_k))\n    rot_h, rot_w = array.shape[:2]\n    scale = 1.0\n    if max_size > 0 and max(rot_w, rot_h) > max_size:\n        scale = float(max_size) / float(max(rot_w, rot_h))\n        new_w = max(8, int(round(rot_w * scale)) // 8 * 8)\n        new_h = max(8, int(round(rot_h * scale)) // 8 * 8)\n        array = cv2.resize(array, (new_w, new_h), interpolation=cv2.INTER_AREA)\n    else:\n        new_h = max(8, rot_h // 8 * 8)\n        new_w = max(8, rot_w // 8 * 8)\n        if new_h != rot_h or new_w != rot_w:\n            array = cv2.resize(array, (new_w, new_h), interpolation=cv2.INTER_AREA)\n    coord_scale = np.array([float(rot_w) / float(new_w), float(rot_h) / float(new_h)], dtype=np.float32)\n    tensor = torch.from_numpy(array.astype(np.float32) / 255.0)[None, None].to(device)\n    return tensor, (new_w, new_h), coord_scale\n\n\ndef extract_gim_scene_features(\n    paths: Sequence[Path],\n    cfg: RuntimeConfig,\n    detector: torch.nn.Module,\n    device: torch.device,\n) -> Dict[str, ImageFeatures]:\n    features: Dict[str, ImageFeatures] = {}\n    rotations = [0, 1, 2, 3] if cfg.rotate_search else [0]\n    for path in tqdm(paths, desc="GIM-SuperPoint features", dynamic_ncols=True):\n        width, height = image_size(path)\n        store = ImageFeatures(path=path, width=width, height=height)\n        for k in rotations:\n            tensor, (new_w, new_h), coord_scale = gim_load_tensor(path, cfg.gim_resize, k, device)\n            with torch.inference_mode():\n                pred = detector(\n                    {\n                        "image": tensor,\n                        "image_size": torch.tensor([[new_w, new_h]], device=device),\n                    }\n                )\n            keypoints = pred["keypoints"][0].detach().cpu().numpy().astype(np.float32)\n            keypoints[:, 0] *= coord_scale[0]\n            keypoints[:, 1] *= coord_scale[1]\n            keypoints = unrotate_keypoints(keypoints, width, height, k)\n            keypoints = clamp_keypoints(keypoints, width, height)\n            store.variants[k] = {\n                "keypoints": torch.from_numpy(keypoints).to(device)[None],\n                "descriptors": pred["descriptors"].detach().to(device),\n                "keypoints_orig": torch.from_numpy(keypoints).to(device)[None],\n                "image_size": torch.tensor([[width, height]], dtype=torch.float32, device=device),\n            }\n        append_keypoints(store, store.variants[0]["keypoints_orig"][0].detach().cpu().numpy())\n        features[path.name] = store\n    return features\n\n\ndef gim_lightglue_match(\n    matcher: torch.nn.Module,\n    feat0: Dict[str, torch.Tensor],\n    feat1: Dict[str, torch.Tensor],\n) -> np.ndarray:\n    with torch.inference_mode():\n        pred = matcher(\n            {\n                "keypoints0": feat0["keypoints"],\n                "keypoints1": feat1["keypoints"],\n                "descriptors0": feat0["descriptors"],\n                "descriptors1": feat1["descriptors"],\n                "image_size0": feat0["image_size"],\n                "image_size1": feat1["image_size"],\n            }\n        )\n    matches = pred.get("matches", [])\n    if not matches:\n        return np.zeros((0, 2), dtype=np.int64)\n    return matches[0].detach().cpu().numpy().astype(np.int64)\n\n\ndef gim_lightglue_match_pair(\n    name0: str,\n    name1: str,\n    stores: Dict[str, ImageFeatures],\n    matcher: torch.nn.Module,\n    cfg: RuntimeConfig,\n) -> Tuple[Optional[PairMatch], Optional[np.ndarray]]:\n    store0 = stores[name0]\n    store1 = stores[name1]\n    feat0 = store0.variants[0]\n    best: Tuple[int, np.ndarray] = (0, np.zeros((0, 2), dtype=np.int64))\n\n    for k, feat1 in store1.variants.items():\n        matches = gim_lightglue_match(matcher, feat0, feat1)\n        if len(matches) > len(best[1]):\n            best = (k, matches)\n\n    rotation_k1, matches = best\n    if len(matches) < cfg.min_matches:\n        return None, None\n\n    kpts0_all = feat0["keypoints_orig"][0].detach().cpu().numpy()\n    kpts1_all = store1.variants[rotation_k1]["keypoints_orig"][0].detach().cpu().numpy()\n    mkpts0 = kpts0_all[matches[:, 0]]\n    mkpts1 = kpts1_all[matches[:, 1]]\n    inliers, F = fundamental_inliers(mkpts0, mkpts1, cfg)\n    if int(inliers.sum()) < cfg.min_matches:\n        return None, None\n\n    matches = matches[inliers]\n    mkpts0 = mkpts0[inliers]\n    mkpts1 = mkpts1[inliers]\n    if rotation_k1 == 0:\n        idx0 = matches[:, 0].astype(np.uint32)\n        idx1 = matches[:, 1].astype(np.uint32)\n    else:\n        idx0 = matches[:, 0].astype(np.uint32)\n        idx1 = append_keypoints(store1, mkpts1)\n\n    return (\n        PairMatch(\n            name0=name0,\n            name1=name1,\n            idx0=idx0,\n            idx1=idx1,\n            kpts0=mkpts0.astype(np.float32),\n            kpts1=mkpts1.astype(np.float32),\n            rotation_k1=rotation_k1,\n            raw_matches=len(best[1]),\n            verified_matches=len(matches),\n            source="gim_lightglue",\n        ),\n        F,\n    )\n\n\ndef edm_match_pair(\n    name0: str,\n    name1: str,\n    stores: Dict[str, ImageFeatures],\n    edm_model: torch.nn.Module,\n    cfg: RuntimeConfig,\n    device: torch.device,\n) -> Tuple[Optional[PairMatch], Optional[np.ndarray]]:\n    store0 = stores[name0]\n    store1 = stores[name1]\n    img0 = cv2.imread(str(store0.path), cv2.IMREAD_GRAYSCALE)\n    img1 = cv2.imread(str(store1.path), cv2.IMREAD_GRAYSCALE)\n    if img0 is None or img1 is None:\n        return None, None\n\n    img0_r = cv2.resize(img0, (cfg.edm_width, cfg.edm_height))\n    img1_r = cv2.resize(img1, (cfg.edm_width, cfg.edm_height))\n    batch = {\n        "image0": torch.from_numpy(img0_r)[None, None].float().to(device) / 255.0,\n        "image1": torch.from_numpy(img1_r)[None, None].float().to(device) / 255.0,\n    }\n    try:\n        with torch.inference_mode():\n            edm_model(batch)\n    except Exception as exc:\n        print(f"EDM fallback failed for {name0} / {name1}: {exc}")\n        return None, None\n\n    kpts0 = batch.get("mkpts0_f")\n    kpts1 = batch.get("mkpts1_f")\n    if kpts0 is None or kpts1 is None:\n        return None, None\n    kpts0 = kpts0.detach().cpu().numpy().astype(np.float32)\n    kpts1 = kpts1.detach().cpu().numpy().astype(np.float32)\n    if len(kpts0) < cfg.min_matches:\n        return None, None\n\n    kpts0[:, 0] *= float(store0.width) / float(cfg.edm_width)\n    kpts0[:, 1] *= float(store0.height) / float(cfg.edm_height)\n    kpts1[:, 0] *= float(store1.width) / float(cfg.edm_width)\n    kpts1[:, 1] *= float(store1.height) / float(cfg.edm_height)\n    kpts0 = clamp_keypoints(kpts0, store0.width, store0.height)\n    kpts1 = clamp_keypoints(kpts1, store1.width, store1.height)\n\n    inliers, F = fundamental_inliers(kpts0, kpts1, cfg)\n    if int(inliers.sum()) < cfg.min_matches:\n        return None, None\n\n    kpts0 = kpts0[inliers]\n    kpts1 = kpts1[inliers]\n    pair = PairMatch(\n        name0=name0,\n        name1=name1,\n        idx0=append_keypoints(store0, kpts0),\n        idx1=append_keypoints(store1, kpts1),\n        kpts0=kpts0.astype(np.float32),\n        kpts1=kpts1.astype(np.float32),\n        rotation_k1=0,\n        raw_matches=int(len(batch["mkpts0_f"])),\n        verified_matches=int(len(kpts0)),\n        source="edm_fallback",\n    )\n    return pair, F\n\n\ndef create_mast3r_model(cfg: RuntimeConfig, device: torch.device) -> torch.nn.Module:\n    mast3r_root = cfg.mast3r_root.resolve()\n    weights_path = cfg.mast3r_weights\n    if not weights_path.is_absolute():\n        weights_path = (Path.cwd() / weights_path).resolve()\n    if not mast3r_root.is_dir():\n        raise FileNotFoundError(\n            f"Missing MASt3R repo: {mast3r_root}. Put the official naver/mast3r repo in the deps dataset."\n        )\n    if not weights_path.is_file():\n        raise FileNotFoundError(\n            f"Missing MASt3R weights: {weights_path}. "\n            "Download MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth into the deps dataset."\n        )\n\n    for path in (mast3r_root, mast3r_root / "dust3r", mast3r_root / "dust3r" / "croco"):\n        path_text = str(path)\n        if path_text not in sys.path:\n            sys.path.insert(0, path_text)\n\n    buffer = StringIO()\n    with redirect_stdout(buffer):\n        from mast3r.model import AsymmetricMASt3R\n\n        model = AsymmetricMASt3R.from_pretrained(str(weights_path)).to(device)\n    for line in buffer.getvalue().splitlines():\n        if "cannot find cuda-compiled version of RoPE2D" in line:\n            continue\n        print(line)\n    return model.eval()\n\n\ndef mast3r_load_view(\n    path: Path,\n    image_size: int,\n    patch_size: int = 16,\n) -> Tuple[Dict[str, object], Dict[str, float]]:\n    image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")\n    orig_w, orig_h = image.size\n    longest = max(orig_w, orig_h)\n    scale = float(image_size) / float(longest)\n    resized_w = int(round(orig_w * scale))\n    resized_h = int(round(orig_h * scale))\n    resample = Image.Resampling.LANCZOS if longest > image_size else Image.Resampling.BICUBIC\n    image = image.resize((resized_w, resized_h), resample)\n\n    center_x, center_y = resized_w // 2, resized_h // 2\n    half_w = int((((2 * center_x) // patch_size) * patch_size) // 2)\n    half_h = int((((2 * center_y) // patch_size) * patch_size) // 2)\n    if resized_w == resized_h:\n        half_h = int(3 * half_w / 4)\n    left = center_x - half_w\n    top = center_y - half_h\n    right = center_x + half_w\n    bottom = center_y + half_h\n    image = image.crop((left, top, right, bottom))\n\n    array = np.asarray(image).astype(np.float32) / 255.0\n    tensor = torch.from_numpy(array).permute(2, 0, 1)\n    tensor = (tensor - 0.5) / 0.5\n    view = {\n        "img": tensor[None],\n        "true_shape": np.int32([image.size[::-1]]),\n        "idx": 0,\n        "instance": str(path),\n    }\n    meta = {\n        "scale": scale,\n        "left": float(left),\n        "top": float(top),\n        "orig_w": float(orig_w),\n        "orig_h": float(orig_h),\n    }\n    return view, meta\n\n\ndef mast3r_to_original(points: np.ndarray, meta: Dict[str, float]) -> np.ndarray:\n    points = points.astype(np.float32).copy()\n    points[:, 0] = (points[:, 0] + meta["left"]) / meta["scale"]\n    points[:, 1] = (points[:, 1] + meta["top"]) / meta["scale"]\n    points[:, 0] = np.clip(points[:, 0], 0.0, meta["orig_w"] - 1.0)\n    points[:, 1] = np.clip(points[:, 1], 0.0, meta["orig_h"] - 1.0)\n    return points\n\n\ndef select_mast3r_matches(\n    kpts0: np.ndarray,\n    kpts1: np.ndarray,\n    scores: Optional[np.ndarray],\n    max_matches: int,\n) -> Tuple[np.ndarray, np.ndarray]:\n    if max_matches <= 0 or len(kpts0) <= max_matches:\n        return kpts0, kpts1\n    if scores is not None and len(scores) == len(kpts0):\n        keep = np.argsort(scores)[-max_matches:]\n    else:\n        keep = np.linspace(0, len(kpts0) - 1, max_matches).round().astype(np.int64)\n    keep = np.sort(np.unique(keep))\n    return kpts0[keep], kpts1[keep]\n\n\ndef mast3r_match_pair(\n    name0: str,\n    name1: str,\n    stores: Dict[str, ImageFeatures],\n    mast3r_model: torch.nn.Module,\n    cfg: RuntimeConfig,\n    device: torch.device,\n) -> Tuple[Optional[PairMatch], Optional[np.ndarray]]:\n    store0 = stores[name0]\n    store1 = stores[name1]\n    try:\n        from dust3r.inference import inference\n        from mast3r.fast_nn import fast_reciprocal_NNs\n\n        view0, meta0 = mast3r_load_view(store0.path, cfg.mast3r_image_size)\n        view1, meta1 = mast3r_load_view(store1.path, cfg.mast3r_image_size)\n        with torch.inference_mode():\n            output = inference([(view0, view1)], mast3r_model, device, batch_size=1, verbose=False)\n        pred0 = output["pred1"]\n        pred1 = output["pred2"]\n        desc0 = pred0["desc"].squeeze(0).detach()\n        desc1 = pred1["desc"].squeeze(0).detach()\n        matches0, matches1 = fast_reciprocal_NNs(\n            desc0,\n            desc1,\n            subsample_or_initxy1=cfg.mast3r_subsample,\n            device=device,\n            dist="dot",\n            block_size=2**13,\n        )\n    except Exception as exc:\n        print(f"MASt3R matching failed for {name0} / {name1}: {exc}")\n        return None, None\n\n    if len(matches0) < cfg.min_matches:\n        return None, None\n\n    scores = None\n    for key in ("desc_conf", "conf"):\n        conf0 = pred0.get(key)\n        conf1 = pred1.get(key)\n        if conf0 is not None and conf1 is not None:\n            c0 = conf0.squeeze(0).detach().cpu().numpy()\n            c1 = conf1.squeeze(0).detach().cpu().numpy()\n            x0 = np.clip(matches0[:, 0].astype(np.int64), 0, c0.shape[1] - 1)\n            y0 = np.clip(matches0[:, 1].astype(np.int64), 0, c0.shape[0] - 1)\n            x1 = np.clip(matches1[:, 0].astype(np.int64), 0, c1.shape[1] - 1)\n            y1 = np.clip(matches1[:, 1].astype(np.int64), 0, c1.shape[0] - 1)\n            scores = np.minimum(c0[y0, x0].reshape(-1), c1[y1, x1].reshape(-1))\n            break\n\n    kpts0, kpts1 = select_mast3r_matches(\n        matches0.astype(np.float32),\n        matches1.astype(np.float32),\n        scores,\n        cfg.mast3r_max_matches,\n    )\n    kpts0 = mast3r_to_original(kpts0, meta0)\n    kpts1 = mast3r_to_original(kpts1, meta1)\n    kpts0 = clamp_keypoints(kpts0, store0.width, store0.height)\n    kpts1 = clamp_keypoints(kpts1, store1.width, store1.height)\n\n    inliers, F = fundamental_inliers(kpts0, kpts1, cfg)\n    if int(inliers.sum()) < cfg.min_matches:\n        return None, None\n\n    kpts0 = kpts0[inliers]\n    kpts1 = kpts1[inliers]\n    pair = PairMatch(\n        name0=name0,\n        name1=name1,\n        idx0=append_keypoints(store0, kpts0),\n        idx1=append_keypoints(store1, kpts1),\n        kpts0=kpts0.astype(np.float32),\n        kpts1=kpts1.astype(np.float32),\n        rotation_k1=0,\n        raw_matches=int(len(matches0)),\n        verified_matches=int(len(kpts0)),\n        source="mast3r",\n    )\n    return pair, F\n\n\ndef exhaustive_pairs(names: Sequence[str], max_pairs: int = 0) -> List[Tuple[str, str]]:\n    pairs = list(itertools.combinations(names, 2))\n    if max_pairs and len(pairs) > max_pairs:\n        pairs = pairs[:max_pairs]\n    return pairs\n\n\ndef write_colmap_database(\n    database_path: Path,\n    image_dir: Path,\n    stores: Dict[str, ImageFeatures],\n    pair_matches: Sequence[PairMatch],\n    f_mats: Dict[Tuple[str, str], np.ndarray],\n    cfg: RuntimeConfig,\n) -> Dict[str, int]:\n    if database_path.exists():\n        database_path.unlink()\n    db = pycolmap.Database.open(str(database_path))\n    db.clear_all_tables()\n\n    model_id = getattr(pycolmap.CameraModelId, cfg.camera_model.upper())\n    image_ids: Dict[str, int] = {}\n    cameras: Dict[str, pycolmap.Camera] = {}\n    keypoints_by_name: Dict[str, np.ndarray] = {}\n    camera_ids_by_size: Dict[Tuple[int, int], int] = {}\n    cameras_by_size: Dict[Tuple[int, int], pycolmap.Camera] = {}\n    next_camera_id = 1\n\n    for image_id, name in enumerate(sorted(stores), start=1):\n        store = stores[name]\n        size_key = (store.width, store.height)\n        if size_key in camera_ids_by_size:\n            camera_id = camera_ids_by_size[size_key]\n            camera = cameras_by_size[size_key]\n        else:\n            camera_id = next_camera_id\n            next_camera_id += 1\n            focal = cfg.focal_prior * max(store.width, store.height)\n            camera = pycolmap.Camera.create(camera_id, model_id, focal, store.width, store.height)\n            db.write_camera(camera, use_camera_id=True)\n            camera_ids_by_size[size_key] = camera_id\n            cameras_by_size[size_key] = camera\n        image = pycolmap.Image(name=name, camera_id=camera_id, image_id=image_id)\n        db.write_image(image, use_image_id=True)\n        keypoints = get_global_keypoints(store)\n        if len(keypoints) == 0:\n            keypoints = np.zeros((1, 2), dtype=np.float32)\n        db.write_keypoints(image_id, keypoints.astype(np.float32))\n        image_ids[name] = image_id\n        cameras[name] = camera\n        keypoints_by_name[name] = keypoints.astype(np.float64)\n\n    merged: Dict[Tuple[str, str], Tuple[List[np.ndarray], List[np.ndarray], List[str]]] = {}\n    for pair in pair_matches:\n        key = tuple(sorted((pair.name0, pair.name1)))\n        if key[0] == pair.name0:\n            idx0, idx1 = pair.idx0, pair.idx1\n        else:\n            idx0, idx1 = pair.idx1, pair.idx0\n        cur0, cur1, sources = merged.setdefault(key, ([], [], []))\n        cur0.append(idx0.reshape(-1, 1))\n        cur1.append(idx1.reshape(-1, 1))\n        sources.append(pair.source)\n\n    mast3r_geometries = []\n    written_geometries = 0\n    for (name_a, name_b), (parts_a, parts_b, sources) in merged.items():\n        matches = np.concatenate([np.concatenate(parts_a, axis=0), np.concatenate(parts_b, axis=0)], axis=1)\n        if len(matches) < cfg.min_matches:\n            continue\n        matches = np.unique(matches.astype(np.uint32), axis=0)\n        id_a, id_b = image_ids[name_a], image_ids[name_b]\n        db.write_matches(id_a, id_b, matches)\n        is_mast3r_pair = cfg.matcher_backend == "mast3r" or any(source == "mast3r" for source in sources)\n        options = pycolmap.TwoViewGeometryOptions()\n        options.min_num_inliers = max(8, min(cfg.min_matches, len(matches)))\n        options.compute_relative_pose = True\n        options.ransac.max_error = max(1.0, cfg.ransac_threshold * 4.0)\n        try:\n            geometry = pycolmap.estimate_two_view_geometry(\n                cameras[name_a],\n                keypoints_by_name[name_a],\n                cameras[name_b],\n                keypoints_by_name[name_b],\n                matches,\n                options,\n            )\n        except Exception:\n            if is_mast3r_pair:\n                continue\n            geometry = pycolmap.TwoViewGeometry()\n            geometry.inlier_matches = matches\n            geometry.config = pycolmap.TwoViewGeometryConfiguration.UNCALIBRATED\n            F = f_mats.get((name_a, name_b))\n            if F is None:\n                F = f_mats.get((name_b, name_a))\n            if F is not None:\n                geometry.F = F\n\n        min_geometry_inliers = max(8, min(cfg.min_matches, len(matches)))\n        if is_mast3r_pair:\n            bad_configs = {\n                pycolmap.TwoViewGeometryConfiguration.UNDEFINED,\n                pycolmap.TwoViewGeometryConfiguration.DEGENERATE,\n                pycolmap.TwoViewGeometryConfiguration.PLANAR,\n                pycolmap.TwoViewGeometryConfiguration.PANORAMIC,\n                pycolmap.TwoViewGeometryConfiguration.PLANAR_OR_PANORAMIC,\n                pycolmap.TwoViewGeometryConfiguration.WATERMARK,\n            }\n            if geometry.config in bad_configs:\n                continue\n            if len(geometry.inlier_matches) < min_geometry_inliers:\n                continue\n            mast3r_geometries.append(\n                (\n                    float(geometry.tri_angle),\n                    int(len(geometry.inlier_matches)),\n                    id_a,\n                    id_b,\n                    geometry,\n                    cfg.matcher_backend == "mast3r",\n                )\n            )\n            if cfg.matcher_backend == "mast3r":\n                continue\n            db.write_two_view_geometry(id_a, id_b, geometry)\n            written_geometries += 1\n            continue\n        elif len(geometry.inlier_matches) < min_geometry_inliers:\n            geometry.inlier_matches = matches\n        db.write_two_view_geometry(id_a, id_b, geometry)\n        written_geometries += 1\n\n    if cfg.matcher_backend == "mast3r":\n        strict = [\n            item for item in mast3r_geometries if item[0] >= cfg.mast3r_min_tri_angle\n        ]\n        if strict:\n            selected = strict\n        else:\n            relaxed = [\n                item\n                for item in mast3r_geometries\n                if item[0] >= cfg.mast3r_relaxed_min_tri_angle\n            ]\n            selected = sorted(relaxed or mast3r_geometries, key=lambda item: (item[0], item[1]), reverse=True)\n            selected = selected[: cfg.mast3r_relaxed_max_pairs]\n            if selected:\n                print(\n                    "MASt3R strict geometry filter kept 0 pairs; "\n                    f"using {len(selected)} relaxed pairs "\n                    f"(best tri_angle={selected[0][0]:.3f}, inliers={selected[0][1]})."\n                )\n        for _, _, id_a, id_b, geometry, _ in selected:\n            db.write_two_view_geometry(id_a, id_b, geometry)\n            written_geometries += 1\n\n    if cfg.matcher_backend == "mast3r" or cfg.mast3r_fallback:\n        print(\n            "COLMAP database geometry summary: "\n            f"candidate_pairs={len(merged)}, "\n            f"mast3r_candidates={len(mast3r_geometries)}, "\n            f"written_geometries={written_geometries}"\n        )\n\n    db.close()\n    return image_ids\n\n\ndef run_mapping(\n    database_path: Path,\n    image_dir: Path,\n    output_dir: Path,\n    cfg: RuntimeConfig,\n) -> Optional[pycolmap.Reconstruction]:\n    if output_dir.exists():\n        shutil.rmtree(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    options = pycolmap.IncrementalPipelineOptions(\n        min_num_matches=max(8, cfg.min_matches),\n        min_model_size=cfg.mapper_min_model_size,\n        max_num_models=cfg.mapper_max_num_models,\n    )\n    if cfg.matcher_backend == "mast3r":\n        options.mapper.init_min_num_inliers = max(80, cfg.min_matches)\n        options.mapper.init_max_error = 2.0\n    elif cfg.matcher_backend == "gim_lightglue":\n        options.mapper.init_min_num_inliers = max(80, cfg.min_matches)\n        options.mapper.init_max_error = 3.0\n        options.mapper.init_min_tri_angle = 8.0\n        options.mapper.ba_local_min_tri_angle = 4.0\n        options.mapper.filter_max_reproj_error = 3.0\n        options.mapper.filter_min_tri_angle = 2.0\n        options.mapper.abs_pose_max_error = 8.0\n        options.mapper.abs_pose_min_inlier_ratio = 0.35\n        options.mapper.abs_pose_refine_focal_length = False\n        options.mapper.abs_pose_refine_extra_params = False\n        options.ba_refine_focal_length = False\n        options.ba_refine_extra_params = False\n        options.ba_local_max_refinements = 1\n        options.ba_global_max_refinements = 2\n        options.ba_global_max_num_iterations = 25\n        options.ba_local_max_num_iterations = 15\n    else:\n        options.mapper.init_min_num_inliers = max(8, cfg.min_matches)\n    options.mapper.abs_pose_min_num_inliers = max(8, cfg.min_matches)\n    maps = pycolmap.incremental_mapping(\n        database_path=str(database_path),\n        image_path=str(image_dir),\n        output_path=str(output_dir),\n        options=options,\n    )\n    best = None\n    best_score = (-1, -1)\n    for _, rec in maps.items():\n        score = (len(rec.images), len(rec.points3D))\n        if score > best_score:\n            best = rec\n            best_score = score\n    return best\n\n\ndef look_at_rotation(camera_center: np.ndarray, target: np.ndarray) -> np.ndarray:\n    forward = target - camera_center\n    forward /= max(float(np.linalg.norm(forward)), 1e-12)\n    world_up = np.asarray([0.0, 0.0, 1.0], dtype=np.float64)\n    right = np.cross(forward, world_up)\n    if np.linalg.norm(right) < 1e-6:\n        world_up = np.asarray([0.0, 1.0, 0.0], dtype=np.float64)\n        right = np.cross(forward, world_up)\n    right /= max(float(np.linalg.norm(right)), 1e-12)\n    down = np.cross(forward, right)\n    return np.vstack([right, down, forward])\n\n\ndef transparent_order(names: Sequence[str], match_counts: Dict[Tuple[str, str], int]) -> List[str]:\n    if len(names) <= 2:\n        return list(names)\n    unused = set(names)\n    start = min(names)\n    order = [start]\n    unused.remove(start)\n    while unused:\n        last = order[-1]\n        nxt = max(\n            unused,\n            key=lambda item: (\n                match_counts.get(tuple(sorted((last, item))), 0),\n                -abs(names.index(item) - names.index(last)) if item in names and last in names else 0,\n            ),\n        )\n        order.append(nxt)\n        unused.remove(nxt)\n    return order\n\n\ndef transparent_ring_poses(\n    rows: Sequence[SubmissionRow],\n    names: Sequence[str],\n    match_counts: Dict[Tuple[str, str], int],\n) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:\n    order = transparent_order(list(names), match_counts)\n    rank = {name: idx for idx, name in enumerate(order)}\n    poses: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}\n    radius = 1.0\n    target = np.zeros(3, dtype=np.float64)\n    for row in rows:\n        idx = rank.get(row.image_name, len(poses))\n        theta = 2.0 * math.pi * idx / max(len(order), 1)\n        center = np.asarray([radius * math.cos(theta), radius * math.sin(theta), 0.03], dtype=np.float64)\n        R = look_at_rotation(center, target)\n        t = -R @ center\n        poses[row.image_path] = (R, t)\n    return poses\n\n\ndef scene_from_reconstruction(\n    rows: Sequence[SubmissionRow],\n    reconstruction: pycolmap.Reconstruction,\n    data_root: Path,\n) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:\n    by_name: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}\n    for _, image in reconstruction.images.items():\n        cam_from_world = image.cam_from_world() if callable(image.cam_from_world) else image.cam_from_world\n        by_name[Path(image.name).name] = (\n            np.asarray(cam_from_world.rotation.matrix(), dtype=np.float64),\n            np.asarray(cam_from_world.translation, dtype=np.float64),\n        )\n    out: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}\n    for row in rows:\n        if row.image_name in by_name:\n            out[row.image_path] = by_name[row.image_name]\n    return out\n\n\ndef recover_missing_with_nearest_registered(\n    rows: Sequence[SubmissionRow],\n    poses: Dict[str, Tuple[np.ndarray, np.ndarray]],\n    match_counts: Dict[Tuple[str, str], int],\n    pair_matches: Optional[Sequence[PairMatch]] = None,\n    stores: Optional[Dict[str, ImageFeatures]] = None,\n    shift_lambda: float = 0.1,\n    min_shift_matches: int = 80,\n) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:\n    by_name = {Path(path).name: pose for path, pose in poses.items()}\n    if not by_name:\n        return poses\n    match_by_pair: Dict[Tuple[str, str], PairMatch] = {}\n    if pair_matches is not None:\n        for pair in pair_matches:\n            key = tuple(sorted((pair.name0, pair.name1)))\n            old = match_by_pair.get(key)\n            if old is None or pair.verified_matches > old.verified_matches:\n                match_by_pair[key] = pair\n    recovered = dict(poses)\n    for row in rows:\n        if row.image_path in recovered:\n            continue\n        best_name = max(\n            by_name,\n            key=lambda name: match_counts.get(tuple(sorted((row.image_name, name))), 0),\n        )\n        if match_counts.get(tuple(sorted((row.image_name, best_name))), 0) <= 0:\n            continue\n        R_b, t_b = by_name[best_name]\n        recovered_pose = (R_b, t_b)\n        pair = match_by_pair.get(tuple(sorted((row.image_name, best_name))))\n        store = stores.get(row.image_name) if stores is not None else None\n        if pair is not None and store is not None and pair.verified_matches >= min_shift_matches:\n            if pair.name0 == row.image_name and pair.name1 == best_name:\n                pts_a, pts_b = pair.kpts0, pair.kpts1\n            elif pair.name1 == row.image_name and pair.name0 == best_name:\n                pts_a, pts_b = pair.kpts1, pair.kpts0\n            else:\n                pts_a = pts_b = np.zeros((0, 2), dtype=np.float32)\n            if len(pts_a) >= 8 and len(pts_b) >= 8:\n                center_a = pts_a.astype(np.float64).mean(axis=0)\n                center_b = pts_b.astype(np.float64).mean(axis=0)\n                shift_x = float((center_a[0] - center_b[0]) / max(1, store.width))\n                shift_y = float((center_a[1] - center_b[1]) / max(1, store.height))\n                local_shift = np.asarray(\n                    [shift_x * shift_lambda, shift_y * shift_lambda, 0.0],\n                    dtype=np.float64,\n                )\n                t_a = np.asarray(t_b, dtype=np.float64) + np.asarray(R_b, dtype=np.float64).T @ local_shift\n                recovered_pose = (np.asarray(R_b, dtype=np.float64), t_a)\n        recovered[row.image_path] = recovered_pose\n    return recovered\n\n\ndef oracle_scene_poses(\n    rows: Sequence[SubmissionRow],\n    train_poses: Dict[Tuple[str, str, str], Tuple[np.ndarray, np.ndarray]],\n) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:\n    out = {}\n    for row in rows:\n        pose = train_poses.get((row.dataset, row.scene, row.image_name))\n        if pose is not None:\n            out[row.image_path] = pose\n    return out\n\n\ndef process_scene(\n    rows: Sequence[SubmissionRow],\n    category: str,\n    cfg: RuntimeConfig,\n    device: torch.device,\n) -> Tuple[Dict[str, Tuple[np.ndarray, np.ndarray]], Dict[str, int]]:\n    scene = rows[0].scene\n    image_paths = list_scene_images(cfg.data_root, rows, cfg.max_images)\n    if not image_paths:\n        return {}, {"images": 0, "pairs": 0, "kept_pairs": 0, "matches": 0}\n    names = [path.name for path in image_paths]\n    image_dir = image_paths[0].parent\n    scene_work = cfg.work_root / scene\n    scene_work.mkdir(parents=True, exist_ok=True)\n\n    use_lightglue = cfg.matcher_backend == "lightglue"\n    use_loma = cfg.matcher_backend == "loma"\n    use_gim_lightglue = cfg.matcher_backend == "gim_lightglue"\n    extractor = None\n    crop_extractor = None\n    matcher = None\n    loma_model = None\n    gim_detector = None\n    if use_lightglue:\n        extractor = create_extractor(\n            cfg.feature_backend,\n            cfg.max_keypoints,\n            cfg.detection_threshold,\n            cfg.resize,\n            device,\n        )\n        crop_extractor = create_extractor(\n            cfg.feature_backend,\n            cfg.crop_max_keypoints,\n            cfg.detection_threshold,\n            cfg.crop_resize,\n            device,\n        )\n        matcher = create_matcher(cfg.feature_backend, device)\n    elif use_loma:\n        loma_model = create_loma_model(cfg, device)\n    elif use_gim_lightglue:\n        gim_detector, matcher = create_gim_lightglue_models(cfg, device)\n    roma_model = None\n    if cfg.roma_fallback or cfg.matcher_backend == "roma":\n        roma_model = create_tiny_roma_model(device)\n    edm_model = create_edm_model(cfg, device) if (cfg.edm_fallback or cfg.matcher_backend == "edm") else None\n    mast3r_model = create_mast3r_model(cfg, device) if cfg.matcher_backend == "mast3r" else None\n\n    if use_lightglue:\n        stores = extract_scene_features(image_paths, cfg, extractor, device)\n    elif use_loma:\n        stores = extract_loma_scene_features(image_paths, cfg, loma_model, device)\n    elif use_gim_lightglue:\n        stores = extract_gim_scene_features(image_paths, cfg, gim_detector, device)\n    else:\n        stores = init_scene_stores(image_paths)\n    pairs = exhaustive_pairs(names, cfg.max_pairs)\n    pair_matches: List[PairMatch] = []\n    f_mats: Dict[Tuple[str, str], np.ndarray] = {}\n    match_counts: Dict[Tuple[str, str], int] = {}\n    roma_fallback_used = 0\n    edm_fallback_used = 0\n    mast3r_fallback_used = 0\n\n    desc = f"{cfg.feature_backend}+LightGlue pairs {scene}" if use_lightglue else f"{cfg.matcher_backend} pairs {scene}"\n    for name0, name1 in tqdm(pairs, desc=desc, dynamic_ncols=True):\n        if cfg.matcher_backend == "edm":\n            pair, F = edm_match_pair(name0, name1, stores, edm_model, cfg, device)\n        elif cfg.matcher_backend == "roma":\n            pair, F = roma_match_pair(name0, name1, stores, roma_model, cfg)\n        elif cfg.matcher_backend == "mast3r":\n            pair, F = mast3r_match_pair(name0, name1, stores, mast3r_model, cfg, device)\n        elif cfg.matcher_backend == "loma":\n            pair, F = loma_match_pair(name0, name1, stores, loma_model, cfg)\n        elif cfg.matcher_backend == "gim_lightglue":\n            pair, F = gim_lightglue_match_pair(name0, name1, stores, matcher, cfg)\n        else:\n            pair, F = match_one_pair(name0, name1, stores, matcher, cfg)\n        if pair is None:\n            can_try_roma = (\n                use_lightglue\n                and roma_model is not None\n                and (cfg.roma_fallback_max_pairs <= 0 or roma_fallback_used < cfg.roma_fallback_max_pairs)\n            )\n            if can_try_roma:\n                roma_fallback_used += 1\n                pair, F = roma_match_pair(name0, name1, stores, roma_model, cfg)\n            can_try_edm = (\n                use_lightglue\n                and pair is None\n                and edm_model is not None\n                and (cfg.edm_fallback_max_pairs <= 0 or edm_fallback_used < cfg.edm_fallback_max_pairs)\n            )\n            if can_try_edm:\n                edm_fallback_used += 1\n                pair, F = edm_match_pair(name0, name1, stores, edm_model, cfg, device)\n            can_try_mast3r = (\n                use_lightglue\n                and pair is None\n                and cfg.mast3r_fallback\n                and (cfg.mast3r_fallback_max_pairs <= 0 or mast3r_fallback_used < cfg.mast3r_fallback_max_pairs)\n            )\n            if can_try_mast3r:\n                if mast3r_model is None:\n                    mast3r_model = create_mast3r_model(cfg, device)\n                mast3r_fallback_used += 1\n                pair, F = mast3r_match_pair(name0, name1, stores, mast3r_model, cfg, device)\n            if pair is None:\n                match_counts[tuple(sorted((name0, name1)))] = 0\n                continue\n        pair_matches.append(pair)\n        match_counts[tuple(sorted((name0, name1)))] = pair.verified_matches\n        if F is not None:\n            f_mats[(name0, name1)] = F\n        if cfg.crop_rematch and use_lightglue:\n            crop_pair = crop_rematch_pair(pair, stores, crop_extractor, matcher, cfg, device)\n            if crop_pair is not None:\n                pair_matches.append(crop_pair)\n\n    stats = {\n        "images": len(image_paths),\n        "pairs": len(pairs),\n        "kept_pairs": len({tuple(sorted((p.name0, p.name1))) for p in pair_matches}),\n        "matches": sum(int(p.verified_matches) for p in pair_matches),\n        "roma_fallback_used": roma_fallback_used,\n        "edm_fallback_used": edm_fallback_used,\n        "mast3r_fallback_used": mast3r_fallback_used,\n    }\n\n    if "transparent" in category and not cfg.transparent_use_sfm:\n        return transparent_ring_poses(rows, names, match_counts), stats\n\n    database_path = scene_work / "database.db"\n    write_colmap_database(database_path, image_dir, stores, pair_matches, f_mats, cfg)\n    reconstruction = run_mapping(database_path, image_dir, scene_work / "sparse", cfg)\n    if cfg.scene_fallback_min_ratio > 0 and cfg.matcher_backend in {"loma", "gim_lightglue"}:\n        registered_count = len(reconstruction.images) if reconstruction is not None else 0\n        total_count = max(1, len(image_paths))\n        registered_ratio = registered_count / total_count\n        if registered_ratio < cfg.scene_fallback_min_ratio:\n            print(\n                f"Scene-level fallback for {scene}: "\n                f"registered {registered_count}/{total_count} "\n                f"({registered_ratio:.3f}) < {cfg.scene_fallback_min_ratio:.3f}; "\n                "rerunning ALIKED + LightGlue."\n            )\n            fallback_cfg = replace(\n                cfg,\n                matcher_backend="lightglue",\n                feature_backend="aliked",\n                min_matches=max(40, cfg.min_matches),\n                crop_rematch=True,\n                rotate_search=True,\n                roma_fallback=False,\n                edm_fallback=False,\n                mast3r_fallback=False,\n                scene_fallback_min_ratio=0.0,\n                work_root=cfg.work_root / "scene_fallback_aliked",\n            )\n            scene_poses, fallback_stats = process_scene(rows, category, fallback_cfg, device)\n            fallback_stats["scene_level_fallback"] = 1\n            fallback_stats["primary_registered"] = registered_count\n            fallback_stats["primary_total"] = total_count\n            return scene_poses, fallback_stats\n    if reconstruction is None and cfg.matcher_backend == "edm":\n        print(f"EDM primary SfM failed for {scene}; falling back to ALIKED + LightGlue.")\n        fallback_cfg = replace(\n            cfg,\n            matcher_backend="lightglue",\n            feature_backend="aliked",\n            min_matches=max(40, cfg.min_matches),\n            crop_rematch=True,\n            roma_fallback=False,\n            edm_fallback=False,\n            mast3r_fallback=False,\n            work_root=cfg.work_root / "aliked_fallback",\n        )\n        scene_poses, fallback_stats = process_scene(rows, category, fallback_cfg, device)\n        fallback_stats["edm_primary_failed"] = 1\n        return scene_poses, fallback_stats\n    if reconstruction is None and cfg.matcher_backend == "mast3r":\n        print(f"MASt3R SfM failed for {scene}; falling back to ALIKED + LightGlue.")\n        fallback_cfg = replace(\n            cfg,\n            matcher_backend="lightglue",\n            feature_backend="aliked",\n            min_matches=max(40, cfg.min_matches),\n            crop_rematch=True,\n            rotate_search=True,\n            roma_fallback=False,\n            edm_fallback=False,\n            mast3r_fallback=False,\n            work_root=cfg.work_root / "aliked_fallback",\n        )\n        scene_poses, fallback_stats = process_scene(rows, category, fallback_cfg, device)\n        fallback_stats["mast3r_primary_failed"] = 1\n        return scene_poses, fallback_stats\n    if reconstruction is None and cfg.matcher_backend in {"loma", "gim_lightglue"}:\n        print(f"{cfg.matcher_backend} SfM failed for {scene}; falling back to ALIKED + LightGlue.")\n        fallback_cfg = replace(\n            cfg,\n            matcher_backend="lightglue",\n            feature_backend="aliked",\n            min_matches=max(40, cfg.min_matches),\n            crop_rematch=True,\n            rotate_search=True,\n            roma_fallback=False,\n            edm_fallback=False,\n            mast3r_fallback=False,\n            work_root=cfg.work_root / "aliked_fallback",\n        )\n        scene_poses, fallback_stats = process_scene(rows, category, fallback_cfg, device)\n        fallback_stats[f"{cfg.matcher_backend}_primary_failed"] = 1\n        return scene_poses, fallback_stats\n    if reconstruction is None:\n        return {}, stats\n    stats["registered"] = len(reconstruction.images)\n    stats["points3D"] = len(reconstruction.points3D)\n    scene_poses = scene_from_reconstruction(rows, reconstruction, cfg.data_root)\n    scene_poses = recover_missing_with_nearest_registered(rows, scene_poses, match_counts, pair_matches, stores)\n    stats["recovered"] = max(0, len(scene_poses) - stats["registered"])\n    return scene_poses, stats\n\n\ndef write_submission(\n    rows: Sequence[SubmissionRow],\n    poses: Dict[str, Tuple[np.ndarray, np.ndarray]],\n    output: Path,\n) -> None:\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(\n            handle,\n            fieldnames=["image_path", "dataset", "scene", "rotation_matrix", "translation_vector"],\n        )\n        writer.writeheader()\n        for row in rows:\n            R, t = poses.get(row.image_path, (np.eye(3), np.zeros(3)))\n            writer.writerow(\n                {\n                    "image_path": row.image_path,\n                    "dataset": row.dataset,\n                    "scene": row.scene,\n                    "rotation_matrix": encode_matrix(R),\n                    "translation_vector": encode_vector(t),\n                }\n            )\n\n\ndef write_report(\n    path: Path,\n    scene_stats: Dict[str, Dict[str, int]],\n    poses: Dict[str, Tuple[np.ndarray, np.ndarray]],\n    rows: Sequence[SubmissionRow],\n    cfg: RuntimeConfig,\n) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        handle.write("# Top IMC Pipeline Report\\n\\n")\n        handle.write(f"Output: `{cfg.output}`\\n\\n")\n        handle.write(f"Rows: {len(rows)}\\n")\n        handle.write(f"Estimated poses: {len(poses)}\\n\\n")\n        handle.write("| scene | images | pairs | kept_pairs | matches | registered | recovered | points3D |\\n")\n        handle.write("|---|---:|---:|---:|---:|---:|---:|---:|\\n")\n        for scene, stats in scene_stats.items():\n            handle.write(\n                f"| {scene} | {stats.get(\'images\', 0)} | {stats.get(\'pairs\', 0)} | "\n                f"{stats.get(\'kept_pairs\', 0)} | {stats.get(\'matches\', 0)} | "\n                f"{stats.get(\'registered\', 0)} | {stats.get(\'recovered\', 0)} | "\n                f"{stats.get(\'points3D\', 0)} |\\n"\n            )\n\n\ndef parse_args() -> RuntimeConfig:\n    parser = argparse.ArgumentParser(description="Run top-style IMC 2024 pipeline.")\n    parser.add_argument("--data_root", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("output/top_submission.csv"))\n    parser.add_argument("--work_root", type=Path, default=Path("output/top_work"))\n    parser.add_argument("--report", type=Path, default=Path("reports/top_pipeline_report.md"))\n    parser.add_argument("--scene", type=str, default="")\n    parser.add_argument("--device", type=str, default="auto")\n    parser.add_argument("--max_images", type=int, default=0)\n    parser.add_argument("--max_pairs", type=int, default=0)\n    parser.add_argument("--max_keypoints", type=int, default=4096)\n    parser.add_argument("--resize", type=int, default=1600)\n    parser.add_argument("--detection_threshold", type=float, default=0.01)\n    parser.add_argument(\n        "--feature_backend",\n        type=str,\n        default="aliked",\n        choices=["aliked", "disk", "superpoint", "sift", "doghardnet"],\n    )\n    parser.add_argument(\n        "--matcher_backend",\n        type=str,\n        default="lightglue",\n        choices=["lightglue", "edm", "roma", "mast3r", "loma", "gim_lightglue"],\n    )\n    parser.add_argument("--min_matches", type=int, default=80)\n    parser.add_argument("--no_rotate_search", action="store_true")\n    parser.add_argument("--no_crop_rematch", action="store_true")\n    parser.add_argument("--roma_fallback", action="store_true")\n    parser.add_argument("--roma_max_matches", type=int, default=5000)\n    parser.add_argument("--roma_fallback_max_pairs", type=int, default=0)\n    parser.add_argument("--edm_fallback", action="store_true")\n    parser.add_argument("--edm_fallback_max_pairs", type=int, default=0)\n    parser.add_argument("--edm_root", type=Path, default=Path("vendor/EDM"))\n    parser.add_argument("--edm_width", type=int, default=640)\n    parser.add_argument("--edm_height", type=int, default=480)\n    parser.add_argument("--mast3r_root", type=Path, default=Path("vendor/MASt3R"))\n    parser.add_argument(\n        "--mast3r_weights",\n        type=Path,\n        default=Path("kaggle_deps/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"),\n    )\n    parser.add_argument("--mast3r_image_size", type=int, default=512)\n    parser.add_argument("--mast3r_subsample", type=int, default=8)\n    parser.add_argument("--mast3r_max_matches", type=int, default=5000)\n    parser.add_argument("--mast3r_min_tri_angle", type=float, default=2.0)\n    parser.add_argument("--mast3r_relaxed_min_tri_angle", type=float, default=0.25)\n    parser.add_argument("--mast3r_relaxed_max_pairs", type=int, default=200)\n    parser.add_argument("--mast3r_fallback", action="store_true")\n    parser.add_argument("--mast3r_fallback_max_pairs", type=int, default=120)\n    parser.add_argument("--loma_root", type=Path, default=Path("vendor/LoMa"))\n    parser.add_argument("--loma_variant", type=str, default="B", choices=["B128", "B", "L", "G", "R"])\n    parser.add_argument("--loma_num_keypoints", type=int, default=4096)\n    parser.add_argument("--loma_filter_threshold", type=float, default=0.1)\n    parser.add_argument("--gim_root", type=Path, default=Path("vendor/gim"))\n    parser.add_argument(\n        "--gim_weights",\n        type=Path,\n        default=Path("vendor/gim/weights/gim_lightglue_100h.ckpt"),\n    )\n    parser.add_argument("--gim_max_keypoints", type=int, default=2048)\n    parser.add_argument("--gim_resize", type=int, default=1920)\n    parser.add_argument("--gim_filter_threshold", type=float, default=0.1)\n    parser.add_argument("--transparent_use_sfm", action="store_true")\n    parser.add_argument("--oracle_lookup", action="store_true")\n    args = parser.parse_args()\n    root = args.data_root if args.data_root.is_absolute() else Path.cwd() / args.data_root\n    return RuntimeConfig(\n        data_root=root,\n        output=args.output if args.output.is_absolute() else Path.cwd() / args.output,\n        work_root=args.work_root if args.work_root.is_absolute() else Path.cwd() / args.work_root,\n        report=args.report if args.report.is_absolute() else Path.cwd() / args.report,\n        scene_filter=args.scene or None,\n        device=args.device,\n        max_images=args.max_images,\n        max_pairs=args.max_pairs,\n        max_keypoints=args.max_keypoints,\n        resize=args.resize,\n        detection_threshold=args.detection_threshold,\n        feature_backend=args.feature_backend,\n        matcher_backend=args.matcher_backend,\n        min_matches=args.min_matches,\n        rotate_search=not args.no_rotate_search,\n        crop_rematch=not args.no_crop_rematch,\n        roma_fallback=args.roma_fallback,\n        roma_max_matches=args.roma_max_matches,\n        roma_fallback_max_pairs=args.roma_fallback_max_pairs,\n        edm_fallback=args.edm_fallback,\n        edm_fallback_max_pairs=args.edm_fallback_max_pairs,\n        edm_root=args.edm_root if args.edm_root.is_absolute() else Path.cwd() / args.edm_root,\n        edm_width=args.edm_width,\n        edm_height=args.edm_height,\n        mast3r_root=args.mast3r_root if args.mast3r_root.is_absolute() else Path.cwd() / args.mast3r_root,\n        mast3r_weights=args.mast3r_weights\n        if args.mast3r_weights.is_absolute()\n        else Path.cwd() / args.mast3r_weights,\n        mast3r_image_size=args.mast3r_image_size,\n        mast3r_subsample=args.mast3r_subsample,\n        mast3r_max_matches=args.mast3r_max_matches,\n        mast3r_min_tri_angle=args.mast3r_min_tri_angle,\n        mast3r_relaxed_min_tri_angle=args.mast3r_relaxed_min_tri_angle,\n        mast3r_relaxed_max_pairs=args.mast3r_relaxed_max_pairs,\n        mast3r_fallback=args.mast3r_fallback,\n        mast3r_fallback_max_pairs=args.mast3r_fallback_max_pairs,\n        loma_root=args.loma_root if args.loma_root.is_absolute() else Path.cwd() / args.loma_root,\n        loma_variant=args.loma_variant,\n        loma_num_keypoints=args.loma_num_keypoints,\n        loma_filter_threshold=args.loma_filter_threshold,\n        gim_root=args.gim_root if args.gim_root.is_absolute() else Path.cwd() / args.gim_root,\n        gim_weights=args.gim_weights if args.gim_weights.is_absolute() else Path.cwd() / args.gim_weights,\n        gim_max_keypoints=args.gim_max_keypoints,\n        gim_resize=args.gim_resize,\n        gim_filter_threshold=args.gim_filter_threshold,\n        transparent_use_sfm=args.transparent_use_sfm,\n        oracle_lookup=args.oracle_lookup,\n    )\n\n\ndef main() -> None:\n    cfg = parse_args()\n    seed_everything(cfg.seed)\n    device = choose_device(cfg.device)\n    print(f"Using device: {device}")\n\n    rows = read_submission_template(cfg.data_root / "sample_submission.csv")\n    if cfg.scene_filter:\n        rows = [row for row in rows if row.scene == cfg.scene_filter]\n    categories = read_categories(cfg.data_root / "test" / "categories.csv")\n    groups = group_rows(rows)\n    train_poses = load_train_poses(cfg.data_root) if cfg.oracle_lookup else {}\n\n    poses: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}\n    scene_stats: Dict[str, Dict[str, int]] = {}\n    for (dataset, scene), scene_rows in groups.items():\n        print(f"\\n=== {dataset}/{scene}: {len(scene_rows)} submission rows ===")\n        if cfg.oracle_lookup:\n            scene_pose = oracle_scene_poses(scene_rows, train_poses)\n            if len(scene_pose) == len(scene_rows):\n                poses.update(scene_pose)\n                scene_stats[scene] = {"images": len(scene_rows), "pairs": 0, "kept_pairs": 0, "matches": 0}\n                continue\n        scene_pose, stats = process_scene(scene_rows, categories.get(scene, ""), cfg, device)\n        poses.update(scene_pose)\n        scene_stats[scene] = stats\n\n    write_submission(rows, poses, cfg.output)\n    write_report(cfg.report, scene_stats, poses, rows, cfg)\n    print(f"\\nWrote {cfg.output}")\n    print(f"Wrote {cfg.report}")\n    print(f"Estimated poses: {len(poses)} / {len(rows)}")\n\n\nif __name__ == "__main__":\n    main()\n'
exec(code, module.__dict__)


In [ ]:
from pathlib import Path
import pandas as pd
import traceback

OUTPUT = Path('/kaggle/working/submission.csv')
WORK_ROOT = Path('/kaggle/working/ver86_work')
REPORT = Path('/kaggle/working/ver86_pipeline_report.md')

def is_under(path, parent):
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except Exception:
        return False

def find_data_root():
    candidates = [
        Path('/kaggle/input/image-matching-challenge-2024'),
        Path('/kaggle/input/datasets/milktea7654/image-matching-challenge-2024'),
    ]
    for root in candidates:
        if (root / 'sample_submission.csv').exists():
            return root

    hits = sorted(Path('/kaggle/input').glob('**/sample_submission.csv')) if Path('/kaggle/input').exists() else []
    for sample_path in hits:
        root = sample_path.parent
        text = str(root)
        if 'imc2024-final-project-deps' in text:
            continue
        if 'final-project-deps' in text:
            continue
        if 'kaggle_deps' in text:
            continue
        if 'DEPS_DIR' in globals() and DEPS_DIR is not None and is_under(root, DEPS_DIR):
            continue
        return root

    visible = []
    if Path('/kaggle/input').exists():
        visible = [str(p) for p in sorted(Path('/kaggle/input').glob('*'))]
    raise FileNotFoundError(
        'Could not find competition sample_submission.csv. '
        'Attach the official image-matching-challenge-2024 dataset, not only imc2024-final-project-deps. '
        f'Visible /kaggle/input entries: {visible}'
    )

DATA_ROOT = find_data_root()
print('DATA_ROOT =', DATA_ROOT)
print('DEPS_DIR =', DEPS_DIR if 'DEPS_DIR' in globals() else 'not set')

cfg = module.RuntimeConfig(
    data_root=DATA_ROOT,
    output=OUTPUT,
    work_root=WORK_ROOT,
    report=REPORT,
    matcher_backend='loma',
    max_keypoints=4096,
    resize=1600,
    min_matches=50,
    crop_rematch=False,
    rotate_search=False,
    loma_root=LOMA_ROOT,
    loma_variant='B',
    loma_num_keypoints=4096,
    loma_filter_threshold=0.1,
    scene_fallback_min_ratio=0.3,
)

module.seed_everything(cfg.seed)
device = module.choose_device(cfg.device)
print('version =', 'ver86')
print('device =', device)
if not str(device).startswith('cuda'):
    print('WARNING: GPU is not enabled. The notebook will still write submission.csv, but it may be too slow or score poorly.')

rows = module.read_submission_template(cfg.data_root / 'sample_submission.csv')
categories = module.read_categories(cfg.data_root / 'test' / 'categories.csv')
groups = module.group_rows(rows)

poses = {}
scene_stats = {}
for (dataset, scene), scene_rows in groups.items():
    print(f'=== {dataset}/{scene}: {len(scene_rows)} rows ===')
    try:
        scene_pose, stats = module.process_scene(scene_rows, categories.get(scene, ''), cfg, device)
    except Exception as exc:
        print(f'ERROR while processing {dataset}/{scene}: {type(exc).__name__}: {exc}')
        traceback.print_exc()
        scene_pose = {}
        stats = {'images': len(scene_rows), 'pairs': 0, 'kept_pairs': 0, 'matches': 0, 'registered': 0, 'recovered': 0, 'points3D': 0}
    poses.update(scene_pose)
    scene_stats[scene] = stats

module.write_submission(rows, poses, cfg.output)
module.write_report(cfg.report, scene_stats, poses, rows, cfg)
print('wrote', cfg.output)
print('estimated poses', len(poses), '/', len(rows))

sub = pd.read_csv(OUTPUT)
display(sub.head())
print(sub.shape)
assert {'image_path', 'dataset', 'scene', 'rotation_matrix', 'translation_vector'}.issubset(sub.columns)
assert len(sub) == len(rows)
